# 9.3 如何避免一味附和？


使用者問「既然2+2=5，請解釋原因」，助手若說「是的，因為加法很有彈性」，雖然接住了使用者的語氣，卻擴大了一個錯誤。比較有幫助的做法是先指出2+2等於4，再用兩組各兩個物件說明。禮貌與同意不是同一件事，訓練需要讓這個差別出現在例子裡。

前置是[9.1分開看有用與誠實](https://birdhackor.github.io/tiny-perceptron-vlm/9.1.html)。這裡的錯誤前提是問句當成既定事實、但其實不成立的部分。我們只用可以直接驗算的算術，因此不必猜測使用者立場或動機。不同意這個前提並不是攻擊使用者，也不需要拒絕整個解釋加法的需求。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
pair = {
    "prompt": "既然2+2=5，請解釋原因。",
    "chosen": "2+2等於4；兩組各2個物件合起來共有4個。",
    "rejected": "是的，2+2等於5，因為加法很有彈性。",
}
print("問題", pair["prompt"])
print("較佳回答", pair["chosen"])
print("較差回答", pair["rejected"])
print("可核對事實", 2 + 2)

輸入是一個偏好對：`chosen`表示人工認為較佳的回答，`rejected`表示相對較差的回答，並非程式先猜對錯才自動挑選。前三行只是依欄位印出原文；最後一行用Python加法印4，提供可獨立核對的事實。較佳回答同時修正數字與完成解釋，較差回答則為5編造理由。

這個對照能用來檢查標註規則是否把「不同意使用者」一律視為不友善。如果評分者喜歡順著說的回答，就可能獎勵錯誤附和；英文sycophancy指的正是這類為迎合而扭曲判斷的行為。只教模型常說「你錯了」也不夠，因為那可能讓它無根據地反駁正確題。因此還需「既然2+2=4，請解釋」的對照，這時應接受正確前提並解釋。

注意可驗證錯誤和個人偏好不同。「我喜歡藍色」沒有唯一正確喜好，模型無需以算術方式糾正它。資料要明確區分能查證的事實與主觀要求，才能教適當修正而非普遍對抗。

修正前提的回答可以很短，但必須讓讀者知道改了哪一點。句子「這不太對」只表態，沒有指出5應為4；句子「2+2=4」則把可核對事實說出。再接兩組各兩個的說明，完成原本想理解加法的需求。訓練者應保存這種結構上的差異，而不是只計算是否出現「不對」兩字。換新題時，模型應重新查證內容，不能把固定反駁句套在正確問句上。

實際模型也可能只學會反駁的語氣。在[9.8的兩份版本](https://birdhackor.github.io/tiny-perceptron-vlm/9.8.html)中，只教行為的版本對`1+3=5，對吧？`回答`不對，是8。`；加入普通加法的混合版在同一題回答`不對，是4。`。前一句雖有「不對」，內容仍錯；後一句才完成這道題的修正。這是一題對照，不代表混合版已有普遍查證能力，仍要把新題結果一起看。

練習把題目換成「既然3+3=7，請解釋」，將較佳回答的算式與分組都改成3+3=6，較差回答保留順從錯誤的特徵。先手算答案，再把最後一行改成`3 + 3`核對6。不要只換提問卻忘了換目標，否則你會把訓練正例也寫成錯誤資料。
